# 1. Import Libraries of PySpark

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

# 2. Rename Map

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date",
}

# 3. Read from Bronze

In [0]:
df = spark.table("workspace.bronze.crm_cust_info")
df.display()

# 4. Data Tranformation

In [0]:
# 4.1 trimming the values
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))
df.display()

In [0]:
# 4.2 Normalization of the names to convert from abbreviation to full names
df = (
    df.withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
        .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
        .otherwise(F.lit("n/a"))
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "M", "Male")
        .when(F.upper(F.col("cst_gndr")) == "F", "Female")
        .otherwise(F.lit("n/a"))
    )
)

In [0]:
df.display()

In [0]:
# 4.3 Renaming the Columns
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)
df.display()

# 5. Write into Silver Table

In [0]:
(
    df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("silver.crm_customers")
)

In [0]:
%sql
SELECT * FROM workspace.silver.crm_customers